<a href="https://colab.research.google.com/github/Alamsyah-WM/Predict-DNA-binding-protein-with-ML-and-DL/blob/main/BIoInformatic_Group1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Setup
Installing pytorch, Transformer, Matplotlib, and sckit-learn

In [1]:
!pip -q install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip -q install transformers==4.44.2 scikit-learn==1.6 matplotlib==3.8.4
!pip -q install fair-esm==2.0.0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 45.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 67.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 66.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 69.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.1/93.1 kB 2.8 MB/s eta 0:00:00


#Import Library
Importing Library from installed setup

In [2]:
import os, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import types
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score, confusion_matrix
from transformers import AutoTokenizer, AutoModel
import matplotlib.pyplot as plt

#Config
Setup base config including:
*   Set seed into 42
*   Detect device GPU
*   Set Hyperparameter
*   Set pre-trained model
*   Model handling


Training Optimazation (Currently set base-on paper)
- Epoch: 25 (Number of complete passes over the training data)
- Batch_size: 128 (Total samples process before 1 weight update)
- lr: 1e-5 (Control how fast the weight changes for optimizer )
- Weight_decay: 1e-2 (Reduce overfitting with AdamW Optimizer)
- scheduler: step (Pick how learning rate changes during training, step: halve LR every few epoch)
- Step_size: 5 (Require scheduler:step, every step_size epochs)
- gamma: 0.5

Early stopping
- dropout: 0.3 (Prevent overfitting by fraction of neuorons randomly disable during training)
- patience: 0.4 (For early stopping, number epoch to wait after no improvement in validation F1 before halting)

Pretrained Model
- esm_model_name: facebook/esm2_t6_8M (model name from hugging face)
- max_len: 1024 (Max length sequence for tokenizer feed into pretrained model. 1024 Hard liimt for ESM2)
- pooling: mean (condense token embedding into 1 vector per protein)
- save_embedding: true (Save computed ESM2 embedding format .pt)

Model handling
- use_class_weight: True (Apply class-imbalance weighting in loss function)
- checkpoint_dir: checkpoints (Directory save .pt)
- model_type: (Pick classification model to train)


In [25]:
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Global config; override per dataset below if needed
base_config = {
    "epochs": 5,
    "batch_size": 8,
    "lr": 1e-4,
    "weight_decay": 0.0,
    "scheduler": None,
    "dropout": 0.3,
    "patience": 1,
    "esm_model_name": "facebook/esm2_t12_35M_UR50D",
    "max_len": 1024,
    "pooling": "mean",
    "save_embeddings": False,
    "embed_batch_size": 8,
    "use_class_weight": False,
    "checkpoint_dir": "checkpoints_min",
}

Device: cpu


#Load Pre-Train model and Embedding Utils
Functions:
- load_esm for loads ESM2 tokenizer and backbone
- seqs_to_embeddings
    - Tokenizes sequences with padding up to max_len.
    - Runs them through ESM2 to obtain last_hidden_state with shape [batch,  length, dim].
    - Pools token embeddings into one vector per sequence (mean, cls, or sum pooling).
    - Moves output back to CPU for caching and reuse.
- Cache embedding
    - If a cached .pt file exists, loads the precomputed embeddings (X), labels (y), and metadata (meta).
    - Otherwise, computes new embeddings, saves them, and returns all objects.
    - Metadata stores the embedding dimension used later for the classifier head.



In [4]:
_tokenizer = None
_backbone = None
_backend = None   # "hf" or "esm"

def load_esm(model_name):
    global _tokenizer, _backbone, _backend
    if _tokenizer is not None and _backbone is not None:
        return _tokenizer, _backbone, _backend
    try:
        _tokenizer = AutoTokenizer.from_pretrained(model_name, do_lower_case=False, trust_remote_code=True)
        _backbone  = AutoModel.from_pretrained(model_name, trust_remote_code=True).eval().to(device)
        _backend = "hf"
        print("[ESM] Loaded via Hugging Face:", model_name)
        return _tokenizer, _backbone, _backend
    except Exception as e:
        print("[ESM] HF load failed:", type(e).__name__, "-", e)
        print("[ESM] Falling back to fair-esm…")
    import esm
    name = model_name.split("/")[-1]
    constructors = {
        "esm2_t6_8M_UR50D":  esm.pretrained.esm2_t6_8M_UR50D,
        "esm2_t12_35M_UR50D":esm.pretrained.esm2_t12_35M_UR50D,
        "esm2_t30_150M_UR50D":esm.pretrained.esm2_t30_150M_UR50D,
        "esm2_t33_650M_UR50D":esm.pretrained.esm2_t33_650M_UR50D,
        "esm2_t36_3B_UR50D": esm.pretrained.esm2_t36_3B_UR50D,
    }
    if name not in constructors:
        raise ValueError(f"Unsupported ESM name for fair-esm fallback: {name}")
    model, alphabet = constructors[name]()
    model = model.eval().to(device)
    Tok = types.SimpleNamespace()
    Tok.batch_converter = alphabet.get_batch_converter()
    Tok.alphabet = alphabet
    _tokenizer = Tok
    _backbone = model
    _backend = "esm"
    print("[ESM] Loaded via fair-esm:", name)
    return _tokenizer, _backbone, _backend

@torch.no_grad()
def seqs_to_embeddings(seqs, model_name, max_len=128, pooling="mean", embed_bs=4):
    tok, mdl, backend = load_esm(model_name)
    embs = []
    for i in range(0, len(seqs), embed_bs):
        batch = seqs[i:i+embed_bs]
        if backend == "hf":
            inputs = tok(batch, padding=True, truncation=True, max_length=max_len, return_tensors="pt").to(device)
            if device.type == "cuda":
                with torch.cuda.amp.autocast(dtype=torch.float16):
                    hidden = mdl(**inputs).last_hidden_state
            else:
                hidden = mdl(**inputs).last_hidden_state
            if pooling == "mean":
                emb = hidden.mean(dim=1)
            elif pooling == "cls":
                emb = hidden[:, 0, :]
            elif pooling == "sum":
                emb = hidden.sum(dim=1)
            else:
                emb = hidden.mean(dim=1)
        else:
            labels, strs, tokens = tok.batch_converter([(str(j), s) for j, s in enumerate(batch)])
            if tokens.size(1) > max_len:
                tokens = torch.cat([tokens[:, :1], tokens[:, 1:max_len], tokens[:, -1:]], dim=1)
            tokens = tokens.to(device)
            if device.type == "cuda":
                with torch.cuda.amp.autocast(dtype=torch.float16):
                    out = mdl(tokens, repr_layers=[mdl.num_layers], return_contacts=False)
            else:
                out = mdl(tokens, repr_layers=[mdl.num_layers], return_contacts=False)
            reps = out["representations"][mdl.num_layers]
            alphabet = tok.alphabet
            mask = (tokens != alphabet.padding_idx) & (tokens != alphabet.cls_idx) & (tokens != alphabet.eos_idx)
            mask = mask.unsqueeze(-1)
            reps_masked = reps * mask
            lengths = mask.sum(dim=1).clamp_min(1)
            emb = reps_masked.sum(dim=1) / lengths
        embs.append(emb.detach().cpu())
        if device.type == "cuda":
            torch.cuda.empty_cache()
    return torch.cat(embs, dim=0)

# quick smoke test
toy = ["ACDEFGHIKL", "MNPQRSTVWY", "AAAAA"]
X_test = seqs_to_embeddings(toy, base_config["esm_model_name"], base_config["max_len"], base_config["pooling"], base_config["embed_batch_size"])
print("Embeddings shape (toy):", tuple(X_test.shape))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/95.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/93.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


config.json:   0%|          | 0.00/775 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/31.4M [00:00<?, ?B/s]

Some weights of EsmModel were not initialized from the model checkpoint at facebook/esm2_t6_8M_UR50D and are newly initialized: ['esm.pooler.dense.bias', 'esm.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


[ESM] Loaded via Hugging Face: facebook/esm2_t6_8M_UR50D
Embeddings shape (toy): (3, 320)


#Load and pre-process dataset
Functions:
- load_csv
    - Normalizes column names to sequence and label.
    - Drops duplicate sequences; if a split column exists, deduplicates within each split.
    - Removes whitespace and ensures labels are strictly 0 or 1
- get_split_from_df
    - Reads provided train and test labels from the dataset.

In [17]:
# ===== Cell 4: Load CSV, clean data, read split =====
def load_and_split_csv(path):
    print(f"\n=== Loading dataset: {path} ===")
    df = pd.read_csv(path)
    rename_map = {}
    for c in df.columns:
        lc = c.lower().strip()
        if lc in ["seq", "sequences", "aa_seq", "protein_sequence"]: rename_map[c] = "sequence"
        elif lc in ["target", "class", "bind", "binding", "is_binding", "y", "label"]: rename_map[c] = "label"
        elif lc in ["split", "set", "dataset"]: rename_map[c] = "set"
    if rename_map: df = df.rename(columns=rename_map)
    assert "sequence" in df.columns, "Missing column: sequence"
    assert "label" in df.columns, "Missing column: label"
    assert "set" in df.columns, "Missing column: set (values: training/test)"
    before = len(df)
    df = df.drop_duplicates(subset=["sequence"]).reset_index(drop=True)
    after = len(df)
    print(f"Removed {before - after} duplicate sequences ({before} → {after})")
    df["sequence"] = df["sequence"].astype(str).str.replace(r"\s+", "", regex=True).str.upper()
    df["label"] = df["label"].astype(int).clip(0, 1)
    df["set"] = df["set"].astype(str).str.lower().str.strip()
    train_df = df[df["set"].isin(["train", "training"])].reset_index(drop=True)
    test_df  = df[df["set"].isin(["test", "testing"])].reset_index(drop=True)
    total = len(df)
    print(f"Train samples: {len(train_df)} ({len(train_df)/total:.2%})")
    print(f"Test samples:  {len(test_df)} ({len(test_df)/total:.2%})")
    print("Label balance (train):", train_df["label"].value_counts().to_dict())
    print("Label balance (test):",  test_df["label"].value_counts().to_dict())
    lens = df["sequence"].str.len()
    print(f"Mean length={lens.mean():.1f}, Max={lens.max()}, >{base_config['max_len']}={(lens>base_config['max_len']).sum()}")
    return train_df, test_df

# --- Quick test on your dataset ---
UNI_CSV = "/content/UniSwiss.csv"
PDB_CSV = "/content/PDB1063-186.csv"

uni_train_df, uni_test_df = load_and_split_csv(UNI_CSV)
print("Train shape:", uni_train_df.shape)
print("Test shape:", uni_test_df.shape)

pdb_train_df, pdb_test_df = load_and_split_csv(PDB_CSV)
print("Train shape:", pdb_train_df.shape)
print("Test shape:", pdb_test_df.shape)



=== Loading dataset: /content/UniSwiss.csv ===
Removed 66 duplicate sequences (9762 → 9696)
Train samples: 8940 (92.20%)
Test samples:  756 (7.80%)
Label balance (train): {0: 4498, 1: 4442}
Label balance (test): {0: 381, 1: 375}
Mean length=478.1, Max=8515, >256=6372
Train shape: (8940, 4)
Test shape: (756, 4)

=== Loading dataset: /content/PDB1063-186.csv ===
Removed 78 duplicate sequences (1249 → 1171)
Train samples: 1060 (90.52%)
Test samples:  111 (9.48%)
Label balance (train): {0: 545, 1: 515}
Label balance (test): {0: 93, 1: 18}
Mean length=243.8, Max=1323, >256=414
Train shape: (1060, 4)
Test shape: (111, 4)


#Embed train/test and build dataloaders

## Embed for UniSwiss

In [18]:
# Embed train/test -> Build loader for Uni
Xtr = seqs_to_embeddings(
    uni_train_df["sequence"].tolist(),
    base_config["esm_model_name"],
    max_len=base_config["max_len"],
    pooling=base_config["pooling"],
    embed_bs=base_config["embed_batch_size"]
)
ytr = torch.tensor(uni_train_df["label"].astype(int).values, dtype=torch.long)

Xte = seqs_to_embeddings(
    uni_test_df["sequence"].tolist(),
    base_config["esm_model_name"],
    max_len=base_config["max_len"],
    pooling=base_config["pooling"],
    embed_bs=base_config["embed_batch_size"]
)
yte = torch.tensor(uni_test_df["label"].astype(int).values, dtype=torch.long)

train_loader = DataLoader(TensorDatasetXY(Xtr, ytr), batch_size=base_config["batch_size"], shuffle=True)
test_loader  = DataLoader(TensorDatasetXY(Xte, yte), batch_size=base_config["batch_size"], shuffle=False)

##Embed for PDB

In [ ]:
Xtr = seqs_to_embeddings(
    pdb_train_df["sequence"].tolist(),
    base_config["esm_model_name"],
    max_len=base_config["max_len"],
    pooling=base_config["pooling"],
    embed_bs=base_config["embed_batch_size"]
)
ytr = torch.tensor(pdb_train_df["label"].astype(int).values, dtype=torch.long)

Xte = seqs_to_embeddings(
    pdb_test_df["sequence"].tolist(),
    base_config["esm_model_name"],
    max_len=base_config["max_len"],
    pooling=base_config["pooling"],
    embed_bs=base_config["embed_batch_size"]
)
yte = torch.tensor(pdb_test_df["label"].astype(int).values, dtype=torch.long)

train_loader2 = DataLoader(TensorDatasetXY(Xtr, ytr), batch_size=base_config["batch_size"], shuffle=True)
test_loader2  = DataLoader(TensorDatasetXY(Xte, yte), batch_size=base_config["batch_size"], shuffle=False)


#Define Model

In [20]:
class BiLSTMHead(nn.Module):
    def __init__(self, in_dim, hidden=128, layers=1, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(1, hidden, layers, batch_first=True, bidirectional=True,
                            dropout=0 if layers==1 else dropout)
        self.fc = nn.Sequential(nn.Dropout(dropout), nn.Linear(hidden*2, 1))
    def forward(self, x):
        x = x.unsqueeze(-1)          # [B, D, 1]
        h, _ = self.lstm(x)          # [B, D, 2H]
        return self.fc(h[:, -1, :]).squeeze(1)

class TransformerHead(nn.Module):
    def __init__(self, in_dim, heads=2, layers=2, ff=256, dropout=0.3):
        super().__init__()
        enc = nn.TransformerEncoderLayer(d_model=in_dim, nhead=heads,
                                         dim_feedforward=ff, dropout=dropout, batch_first=True)
        self.tr = nn.TransformerEncoder(enc, num_layers=layers)
        self.fc = nn.Sequential(nn.Dropout(dropout), nn.Linear(in_dim, 1))
    def forward(self, x):
        x = x.unsqueeze(1)           # [B, 1, D]
        h = self.tr(x)               # [B, 1, D]
        return self.fc(h[:, 0, :]).squeeze(1)

class CNNHead(nn.Module):
    def __init__(self, in_dim, channels=64, kernel_size=3, dropout=0.3):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(1, channels, kernel_size=kernel_size, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(channels, channels * 2, kernel_size=kernel_size, padding=1),
            nn.ReLU(),
            nn.AdaptiveMaxPool1d(1)
        )
        self.fc = nn.Sequential(nn.Dropout(dropout), nn.Linear(channels * 2, 1))
    def forward(self, x):
        x = x.unsqueeze(1)
        h = self.conv(x).squeeze(-1)
        return self.fc(h).squeeze(1)

class MLPHead(nn.Module):
    def __init__(self, in_dim, hidden=[512, 128, 32], dropout=0.3):
        super().__init__()
        layers = []
        last = in_dim
        for h in hidden:
            layers += [nn.Linear(last, h), nn.ReLU(), nn.Dropout(dropout)]
            last = h
        layers += [nn.Linear(last, 1)]
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x).squeeze(1)


def make_head(name, in_dim, dropout):
    if name == "BiLSTM":     return BiLSTMHead(in_dim, hidden=128, layers=1, dropout=dropout).to(device)
    if name == "Transformer": return TransformerHead(in_dim, heads=2, layers=2, ff=256, dropout=dropout).to(device)
    if name == "CNN":         return CNNHead(in_dim, channels=64, kernel_size=3, dropout=dropout).to(device)
    if name == "MLP":         return MLPHead(in_dim, hidden=[512,128,32], dropout=dropout).to(device)
    raise ValueError("Unknown head")

#Train Model

In [21]:
def train_and_eval(HEAD="Transformer", epochs=10, lr=1e-4, dropout=0.3, use_class_weight=True, early_stop_patience=2):
    assert 'train_loader' in globals() and 'test_loader' in globals(), "Run Cell 5 first to build loaders."
    in_dim = Xtr.shape[1]
    model = make_head(HEAD, in_dim, dropout)

    # class weight from train labels (recommended for PDB)
    if use_class_weight:
        y_train_np = train_df["label"].astype(int).values
        neg, pos = (y_train_np == 0).sum(), (y_train_np == 1).sum()
        pos_weight = torch.tensor([neg / max(pos, 1)], device=device, dtype=torch.float32)
    else:
        pos_weight = None

    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.0)

    best_loss, wait, best_state = float("inf"), 0, None
    for ep in range(1, epochs + 1):
        model.train()
        total_loss, n = 0.0, 0
        for xb, yb in train_loader:
            xb = xb.to(device); yb = yb.float().to(device)
            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward(); optimizer.step()
            total_loss += loss.item() * len(yb); n += len(yb)
        tr_loss = total_loss / max(n, 1)
        print(f"Epoch {ep}/{epochs} - train loss: {tr_loss:.4f}")

        # early stop on train loss
        if tr_loss + 1e-6 < best_loss:
            best_loss = tr_loss; wait = 0
            best_state = {k: v.detach().cpu() for k, v in model.state_dict().items()}
        else:
            wait += 1
            if wait >= early_stop_patience:
                print("Early stop on train loss.")
                break

    # load best state
    if best_state is not None:
        model.load_state_dict({k: v.to(device) for k, v in best_state.items()})

    # evaluate
    model.eval(); y_true, y_prob = [], []
    with torch.no_grad():
        for xb, yb in test_loader:
            xb = xb.to(device)
            prob = torch.sigmoid(model(xb)).cpu().numpy()
            y_prob += prob.tolist(); y_true += yb.numpy().tolist()
    y_true = np.array(y_true, dtype=int)
    y_prob = np.array(y_prob, dtype=float)
    y_pred = (y_prob >= 0.5).astype(int)

    acc = accuracy_score(y_true, y_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="binary", zero_division=0)
    try:
        auc = roc_auc_score(y_true, y_prob)
    except Exception:
        auc = float("nan")
    cm = confusion_matrix(y_true, y_pred)

    print("\n=== Test metrics ===")
    print(f"Accuracy:  {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall:    {rec:.4f}")
    print(f"F1-score:  {f1:.4f}")
    print(f"ROC-AUC:   {auc:.4f}")
    print("Confusion matrix:\n", cm)

    return {"acc": acc, "prec": prec, "rec": rec, "f1": f1, "auc": auc, "cm": cm, "head": HEAD}

metrics_uni = train_and_eval(HEAD="Transformer", epochs=10, lr=1e-4, dropout=0.3, use_class_weight=False) #Change your model here!
metrics_pdb = train_and_eval(HEAD="Transformer", epochs=15, lr=1e-4, dropout=0.3, use_class_weight=True, early_stop_patience=3) #Change your model here!


Epoch 1/10 - train loss: 0.4980
Epoch 2/10 - train loss: 0.4269
Epoch 3/10 - train loss: 0.4065
Epoch 4/10 - train loss: 0.3973
Epoch 5/10 - train loss: 0.3857
Epoch 6/10 - train loss: 0.3801
Epoch 7/10 - train loss: 0.3774
Epoch 8/10 - train loss: 0.3744
Epoch 9/10 - train loss: 0.3683
Epoch 10/10 - train loss: 0.3627

=== Test metrics ===
Accuracy:  0.8108
Precision: 0.8152
Recall:    0.8000
F1-score:  0.8075
ROC-AUC:   0.8959
Confusion matrix:
 [[313  68]
 [ 75 300]]
Epoch 1/15 - train loss: 0.4931
Epoch 2/15 - train loss: 0.4298
Epoch 3/15 - train loss: 0.4098
Epoch 4/15 - train loss: 0.3970
Epoch 5/15 - train loss: 0.3913
Epoch 6/15 - train loss: 0.3845
Epoch 7/15 - train loss: 0.3765
Epoch 8/15 - train loss: 0.3732
Epoch 9/15 - train loss: 0.3692
Epoch 10/15 - train loss: 0.3699
Epoch 11/15 - train loss: 0.3606
Epoch 12/15 - train loss: 0.3583
Epoch 13/15 - train loss: 0.3606
Epoch 14/15 - train loss: 0.3578
Epoch 15/15 - train loss: 0.3512

=== Test metrics ===
Accuracy:  0.8135

#Summary result

Epoch 1/5 - train loss: 0.6920
Epoch 2/5 - train loss: 0.6812
Epoch 3/5 - train loss: 0.6707
Epoch 4/5 - train loss: 0.6624
Epoch 5/5 - train loss: 0.6531

=== Test metrics (small run, head = BiLSTM ) ===
acc: 0.5992
prec: 0.6000
rec: 0.5760
f1: 0.5878
auc: 0.6356
Confusion matrix:
 [[237 144]
 [159 216]]
